# HW3c Urban Traffic — Colab (from scratch)

**Runtime → Change runtime type → GPU** (A100 / L4 / T4 all fine), then run top to bottom.

Data layout:
- Parts **2 / 3 / 5**: official BDD `box2d` labels → `data/bdd100k_det_mini`
- Part **4**: 10k semantic masks → `data/bdd100k_mini` + `seg_masks/`

Checkpoints / YOLO runs / results / logs are stored under Drive so a disconnect does not wipe them.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Fresh Drive folder so old pseudo-box caches do not leak in.
DRIVE = '/content/drive/MyDrive/245hw3c_v2'
REPO = '/content/245hw3c'
os.makedirs(DRIVE, exist_ok=True)
print('DRIVE =', DRIVE)

In [ ]:
# Public repo. Optional Colab secret GITHUB_TOKEN if you switch it private again.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

url = (
    f'https://{token}@github.com/ericezhao/245hw3c.git'
    if token else 'https://github.com/ericezhao/245hw3c.git'
)

if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone -q {url} {REPO}
%cd {REPO}
!git log --oneline -3
!git rev-parse --abbrev-ref HEAD

In [ ]:
!pip install -q -r requirements.txt
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# Keep heavy outputs on Drive across reconnects.
for name in ['checkpoints', 'yolo_runs', 'results', 'logs']:
    local = os.path.join(REPO, name)
    remote = os.path.join(DRIVE, name)
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local) or os.path.exists(local):
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            shutil.rmtree(local)
        elif os.path.isfile(local):
            os.remove(local)
    os.symlink(remote, local)
!ls -la | grep -- '->'

## Data

First run downloads everything and caches tarballs on Drive.
Later reconnects unpack from Drive (much faster).

If you need a **full data rebuild**, delete `bdd_det_data.tar` / `bdd_seg_data.tar` in `MyDrive/245hw3c_v2` and re-run the next cell.

In [ ]:
DET_TAR = os.path.join(DRIVE, 'bdd_det_data.tar')
SEG_TAR = os.path.join(DRIVE, 'bdd_seg_data.tar')

# --- official detection boxes (Parts 2 / 3 / 5) ---
if os.path.exists(DET_TAR):
    print('restoring detection mini from Drive...')
    !tar -xf {DET_TAR} -C {REPO}
else:
    print('building detection mini from official BDD labels...')
    !python -u prepare_bdd_det_mini.py
    !tar -cf {DET_TAR} data/bdd100k_det_mini
    print('cached', DET_TAR)

# --- semantic masks (Part 4) ---
if os.path.exists(SEG_TAR):
    print('restoring semantic mini from Drive...')
    !tar -xf {SEG_TAR} -C {REPO}
else:
    print('building semantic mini...')
    !python -u prepare_bdd_mini.py
    !tar -cf {SEG_TAR} data/bdd100k_mini seg_masks
    print('cached', SEG_TAR)

!ls -la data/bdd100k_det_mini/annotations data/bdd100k_mini/annotations seg_masks/train | head
!python - <<'PY'
import json
from pathlib import Path
for split in ('train', 'val'):
    p = Path(f'data/bdd100k_det_mini/annotations/instances_{split}.json')
    d = json.loads(p.read_text())
    print(split, 'images', len(d['images']), 'boxes', len(d['annotations']),
          'desc', d.get('info', {}).get('description', '')[:60])
PY

## Train + evaluate

Order: EDA → YOLO (hits the hard mAP bar first) → Faster R-CNN → DeepLab → Mask R-CNN → report.

In [ ]:
!python -u part1_eda.py 2>&1 | tee logs/part1_eda.log

In [ ]:
# YOLOv8s @ 640, 50 epochs — official boxes
!rm -rf yolo_runs/bdd_yolov8s yolo_bdd
!python -u part3_yolov8.py --model yolov8s.pt --batch 16 --imgsz 640 --epochs 50 2>&1 | tee logs/part3_yolov8.log

In [ ]:
!python -u part2_faster_rcnn.py 2>&1 | tee logs/part2_faster_rcnn.log

In [ ]:
!python -u part4_deeplabv3.py 2>&1 | tee logs/part4_deeplabv3.log

In [ ]:
!python -u part5_mask_rcnn.py 2>&1 | tee logs/part5_mask_rcnn.log

In [ ]:
!python -u part6_analysis.py 2>&1 | tee logs/part6_analysis.log
!cp -f report.pdf {DRIVE}/ 2>/dev/null || true
!ls -la results/
!cat results/metrics_table.csv

## Done

Everything lands in Drive: `MyDrive/245hw3c_v2/` (`results`, `logs`, `checkpoints`, `yolo_runs`, `report.pdf`).

If the runtime dies mid-run: remount Drive, re-run clone → deps → symlink → data (tar restore) cells, then continue from the failed part. Resume logic will pick up finished epochs when a checkpoint exists.